## 0. Imports and Setup


In [ ]:
import os
import sys
import torch
import torch.nn as nn
from pathlib import Path

import numpy as np
from PIL import Image
import torch.nn.functional as F


# Reproducibility
torch.manual_seed(42)
np.random.seed(42)

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print(f"Device: {device}")
if torch.cuda.is_available():
    torch.backends.cudnn.benchmark = True

# Mount Google Drive
from google.colab import drive
drive.mount('/content/drive')

EXPORT_DIR = Path("/content/drive/MyDrive/UN/autoencoder_export")
print("Export dir:", EXPORT_DIR)

sys.path.append(str(EXPORT_DIR))
from autoencoder_model import Autoencoder

WEIGHTS = "autoencoder_best.pth" 
print("Loading weights from:", EXPORT_DIR / WEIGHTS)
model = Autoencoder(latent_dim=128, img_size=128).to(device)

state = torch.load(EXPORT_DIR / WEIGHTS, map_location=device)
model.load_state_dict(state)
model.eval()

print("Model loaded and ready")


In [ ]:
def load_rgba(path, size=128):
    """PNG with alpha, scaled to 128×128."""
    img = Image.open(path).convert("RGBA").resize((size, size), Image.LANCZOS)

    arr = np.asarray(img).astype(np.float32) / 127.5 - 1.0   # [-1, 1]
    arr = np.transpose(arr, (2, 0, 1))                      # HWC → CHW
    tensor = torch.tensor(arr).unsqueeze(0).to(device)       # 1×4×H×W
    return tensor, img

def to_image(t):
    """Tensor [-1,1] → PIL.Image RGBA."""
    t = t.detach().cpu().clamp(-1, 1)
    t = (t[0].permute(1, 2, 0).numpy() + 1) * 127.5
    t = t.astype(np.uint8)
    return Image.fromarray(t, mode="RGBA")

In [ ]:
test_image_path = "/content/test.png"  # <-- wstaw swój plik PNG

x, original_pil = load_rgba(test_image_path)
with torch.no_grad():
    recon, latent = model(x)

recon_pil = to_image(recon)

display(original_pil)
display(recon_pil)

print("Test reconstruction complete")
